### ライブラリインポート

In [ ]:
import json
import sys
import requests
import yaml
import pyarrow
import matplotlib.pyplot as plt

from datetime import date, datetime, timedelta
import time
from dateutil.relativedelta import relativedelta

from IPython.display import display
import pandas as pd
from joblib import Parallel, delayed
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

from playwright.sync_api import sync_playwright
# from playwright.async_api import async_playwright

sys.path.append(str(Path("../src").resolve()))
from create_pdf import html_to_pdf

# from google.oauth2.credentials import Credentials
# from google_auth_oauthlib.flow import InstalledAppFlow
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload

### パラメータ読み込み

In [2]:
config_path = "../input/conf/"

with open(config_path + "credentials.yml", "r") as f:
    credentials = yaml.safe_load(f)

with open(config_path + "parameters.yml", "r") as f:
    params = yaml.safe_load(f)

# J-Quants API関連
API_KEY = credentials["jquants"]["key"]

jquants_base_url = params["jquants"]["base_url"]
Path_equities_master = params["jquants"]["paths"]["equities_master"]
Path_daily_bars = params["jquants"]["paths"]["daily_bars"]
Path_fins_summary = params["jquants"]["paths"]["fins_summary"]
Path_earnings_calender = params["jquants"]["paths"]["earnings_calender"]
Path_markets_calender = params["jquants"]["paths"]["markets_calender"]
Path_investors_type = params["jquants"]["paths"]["investors_type"]
Path_daily_topix = params["jquants"]["paths"]["daily_topix"]

print(jquants_base_url)

# LINE Messaging API関連
LINE_CHANNEL_ACCESS_TOKEN = credentials["LINE_official_account"]["channel_access_token"]

line_base_url = params["LINE"]["base_url"]
Path_broadcast = params["LINE"]["paths"]["broadcast"]

print(line_base_url)

https://api.jquants.com
https://api.line.me


### API呼び出し・データ取得

In [3]:
headers = {"x-api-key": API_KEY}

today = date.today()
from_ = today - timedelta(days=400)
to = today

api_call_params = {}
api_call_params["from"] = from_
api_call_params["to"] = to

print(api_call_params["to"])

2026-05-05


In [4]:
### 最新の営業日を取得
res = requests.get(jquants_base_url + Path_markets_calender, params=api_call_params, headers=headers)
if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    api_call_params["pagination_key"] = d["pagination_key"]
    res = requests.get(jquants_base_url + Path_markets_calender, params=api_call_params, headers=headers)
    d = res.json()
    data += d["data"]
  df = pd.DataFrame(data)
else:
  print(res.json())
  
df["Date"] = pd.to_datetime(df["Date"]).dt.date
market_date_last_400days = df[(df["Date"] <= today) & (df["HolDiv"] == "1")]["Date"]
newest_market_date = max(market_date_last_400days)

# 最新の営業日から数えて直近1年の営業日を取得
market_date_last_365days = list(market_date_last_400days[market_date_last_400days >= newest_market_date - timedelta(days=365)])
print(newest_market_date)
print(market_date_last_365days)

2026-05-01
[datetime.date(2025, 5, 1), datetime.date(2025, 5, 2), datetime.date(2025, 5, 7), datetime.date(2025, 5, 8), datetime.date(2025, 5, 9), datetime.date(2025, 5, 12), datetime.date(2025, 5, 13), datetime.date(2025, 5, 14), datetime.date(2025, 5, 15), datetime.date(2025, 5, 16), datetime.date(2025, 5, 19), datetime.date(2025, 5, 20), datetime.date(2025, 5, 21), datetime.date(2025, 5, 22), datetime.date(2025, 5, 23), datetime.date(2025, 5, 26), datetime.date(2025, 5, 27), datetime.date(2025, 5, 28), datetime.date(2025, 5, 29), datetime.date(2025, 5, 30), datetime.date(2025, 6, 2), datetime.date(2025, 6, 3), datetime.date(2025, 6, 4), datetime.date(2025, 6, 5), datetime.date(2025, 6, 6), datetime.date(2025, 6, 9), datetime.date(2025, 6, 10), datetime.date(2025, 6, 11), datetime.date(2025, 6, 12), datetime.date(2025, 6, 13), datetime.date(2025, 6, 16), datetime.date(2025, 6, 17), datetime.date(2025, 6, 18), datetime.date(2025, 6, 19), datetime.date(2025, 6, 20), datetime.date(2025,

In [5]:
### 直近の営業日の上場銘柄一覧を取得
api_call_params = {}
api_call_params["date"] = newest_market_date

res = requests.get(jquants_base_url + Path_equities_master, params=api_call_params, headers=headers)
if res.status_code == 200:
  d = res.json()
  data = d["data"]
  while "pagination_key" in d:
    api_call_params["pagination_key"] = d["pagination_key"]
    res = requests.get(jquants_base_url + Path_equities_master, params=api_call_params, headers=headers)
    d = res.json()
    data += d["data"]
  equities_master = pd.DataFrame(data)
else:
  print(res.json())

equities_master

,Date,Code,CoName,CoNameEn,S17,S17Nm,S33,S33Nm,ScaleCat,Mkt,MktNm,Mrgn,MrgnNm
0,2026-05-01,13010,極洋,"KYOKUYO CO.,LTD.",1,食品,0050,水産・農林業,TOPIX Small 1,0111,プライム,2,貸借
1,2026-05-01,13050,大和アセットマネジメント株式会社 ｉＦｒｅｅＥＴＦ ＴＯＰＩＸ（年１回決算型）,iFreeETF TOPIX (Yearly Dividend Type),99,その他,9999,その他,-,0109,その他,2,貸借
2,2026-05-01,13060,野村アセットマネジメント株式会社 ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ連動型上場投信,NEXT FUNDS TOPIX Exchange Traded Fund,99,その他,9999,その他,-,0109,その他,2,貸借
3,2026-05-01,13080,アモーヴァ・アセットマネジメント株式会社 上場インデックスファンドＴＯＰＩＸ,Listed Index Fund TOPIX,99,その他,9999,その他,-,0109,その他,2,貸借
4,2026-05-01,13090,野村アセットマネジメント株式会社 ＮＥＸＴ ＦＵＮＤＳ ＣｈｉｎａＡＭＣ・中国株式・上証５０...,NEXT FUNDS ChinaAMC SSE50 Index Exchange Trade...,99,その他,9999,その他,-,0109,その他,2,貸借
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4444,2026-05-01,99910,ジェコス,GECOSS CORPORATION,13,商社・卸売,6050,卸売業,TOPIX Small 2,0111,プライム,2,貸借
4445,2026-05-01,99930,ヤマザワ,"YAMAZAWA CO.,LTD.",14,小売,6100,小売業,-,0112,スタンダード,1,信用
4446,2026-05-01,99940,やまや,YAMAYA CORPORATION,14,小売,6100,小売業,-,0112,スタンダード,2,貸借
4447,2026-05-01,99960,サトー商会,"Satoh&Co.,Ltd.",13,商社・卸売,6050,卸売業,-,0112,スタンダード,2,貸借


In [ ]:
# data = []

# # date = market_date_last_365days[0]

# for date in market_date_last_365days[0:100]:

#   api_call_params = {}
#   api_call_params["date"] = date

#   # 銘柄×日別の株価四本値データ取得
#   res = requests.get(jquants_base_url + Path_daily_bars, params=api_call_params, headers=headers)

#   if res.status_code == 200:
#     d = res.json()
#     data += d["data"]
#     while "pagination_key" in d:
#       api_call_params["pagination_key"] = d["pagination_key"]
#       res = requests.get(jquants_base_url + Path_daily_bars, params=api_call_params, headers=headers)
#       d = res.json()
#       data += d["data"]
#     # df = pd.DataFrame(data)
#     # display(df)
#     # display(data)
#   else:
#     print(res.json())
    
# df = pd.DataFrame(data)
# display(df)

KeyboardInterrupt: 

In [ ]:
### 直近1年の全銘柄の日別株価データを取得
def get_daily_stock_data(date):
  
  print(f"start {date}")
  
  api_call_params = {}
  api_call_params["date"] = date

  # 銘柄×日別の株価四本値データ取得
  res = requests.get(jquants_base_url + Path_daily_bars, params=api_call_params, headers=headers)

  if res.status_code == 200:
    d = res.json()
    extracted_data = d["data"]
    while "pagination_key" in d:
      api_call_params["pagination_key"] = d["pagination_key"]
      res = requests.get(jquants_base_url + Path_daily_bars, params=api_call_params, headers=headers)
      d = res.json()
      extracted_data += d["data"]
    # df = pd.DataFrame(extracted_data)
    # display(df)
    # display(extracted_data)
    
  elif res.status_code == 429:
    wait = int(res.headers.get("Retry-After", 60))
    print(f"Rate limit. Sleeping {wait} seconds...")
    time.sleep(wait)
    
  else:
    print(res.status_code)
    print(res.json())
    
  return extracted_data

# market_date_last_365days = market_date_last_365days[0:100]
# print(market_date_last_365days)

daily_stock_data = Parallel(n_jobs=2)(
  delayed(get_daily_stock_data)(date) for date in market_date_last_365days
)

# display(daily_stock_data)
# df = pd.DataFrame(daily_stock_data)
# display(df)

daily_stock_df = pd.concat(
  (
    pd.DataFrame(trial_result)
    for trial_result in daily_stock_data
  ),
  ignore_index=True
)
daily_stock_df

,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo
0,2025-05-01,13010,4205.0,4205.0,4120.0,4120.0,0,0,34200.0,1.420145e+08,1.0,4205.0,4205.0,4120.0,4120.0,34200.0
1,2025-05-01,13050,2864.5,2880.0,2853.5,2869.5,0,0,113820.0,3.259711e+08,1.0,2864.5,2880.0,2853.5,2869.5,113820.0
2,2025-05-01,13060,2834.0,2850.0,2822.0,2845.0,0,0,2716090.0,7.696015e+09,1.0,283.4,285.0,282.2,284.5,27160900.0
3,2025-05-01,13080,2801.0,2816.0,2789.0,2808.0,0,0,324255.0,9.089513e+08,1.0,2801.0,2816.0,2789.0,2808.0,324255.0
4,2025-05-01,13090,41100.0,41760.0,41100.0,41740.0,0,0,16.0,6.644200e+05,1.0,41100.0,41760.0,41100.0,41740.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1083833,2026-05-01,99910,1611.0,1618.0,1587.0,1599.0,0,0,57300.0,9.194810e+07,1.0,1611.0,1618.0,1587.0,1599.0,57300.0
1083834,2026-05-01,99930,1161.0,1162.0,1155.0,1155.0,0,0,800.0,9.269000e+05,1.0,1161.0,1162.0,1155.0,1155.0,800.0
1083835,2026-05-01,99940,2347.0,2410.0,2326.0,2406.0,0,0,30100.0,7.132890e+07,1.0,2347.0,2410.0,2326.0,2406.0,30100.0
1083836,2026-05-01,99960,2250.0,2250.0,2227.0,2227.0,0,0,400.0,8.935000e+05,1.0,2250.0,2250.0,2227.0,2227.0,400.0


In [6]:
daily_stock_df.to_parquet(f"../output/tmp/daily_stock_data_365days_{newest_market_date}.parquet")

In [6]:
daily_stock_df = pd.read_parquet(f"../output/tmp/daily_stock_data_365days_2026-05-01.parquet")

### Preprocessing

In [30]:
### 直近営業日で52週高値を更新した銘柄を抽出
daily_stock_df["Date"] = pd.to_datetime(daily_stock_df["Date"], format="%Y-%m-%d").dt.date

newest_market_date_stock_df = daily_stock_df[["Date", "Code", "AdjH", "AdjC"]][daily_stock_df["Date"] == newest_market_date].reset_index(drop=True)

high_365days_df = (
    daily_stock_df[["Date", "Code", "AdjH"]].groupby("Code")["AdjH"]
    .max()
    .reset_index()
    .rename(columns={"AdjH": "highest_365days"})
)

merged_df = pd.merge(
    newest_market_date_stock_df,
    high_365days_df,
    how="left",
    on="Code"
)
new_highs_df = merged_df[merged_df["AdjH"] == merged_df["highest_365days"]].reset_index(drop=True)
new_highs_df = pd.merge(
    new_highs_df,
    equities_master[["Code", "CoName", "S17Nm", "MktNm"]],
    how="left",
    on="Code"
)
new_highs_df

,Date,Code,AdjH,AdjC,highest_365days,CoName,S17Nm,MktNm
0,2026-05-01,14070,3090.0,2884.0,3090.0,ウエストホールディングス,建設・資材,スタンダード
1,2026-05-01,19420,7029.0,6734.0,7029.0,関電工,建設・資材,プライム
2,2026-05-01,19460,2486.0,2454.0,2486.0,トーエネック,建設・資材,プライム
3,2026-05-01,20330,81090.0,80500.0,81090.0,ノムラ・ヨーロッパ・ファイナンス・エヌ・ブイ ＮＥＸＴ ＮＯＴＥＳ 韓国ＫＯＳＰＩ・ダブル・...,その他,その他
4,2026-05-01,20870,3309.0,3307.0,3309.0,農林中金全共連アセットマネジメント株式会社 ＮＺＡＭ 上場投信 ＮＡＳＤＡＱ１００（為替ヘッ...,その他,その他
...,...,...,...,...,...,...,...,...
83,2026-05-01,90120,2250.0,2250.0,2250.0,秩父鉄道,運輸・物流,スタンダード
84,2026-05-01,90410,3593.0,3453.0,3593.0,近鉄グループホールディングス,運輸・物流,プライム
85,2026-05-01,94120,3650.0,3515.0,3650.0,スカパーＪＳＡＴ,情報通信・サービスその他,プライム
86,2026-05-01,95190,1150.0,1111.0,1150.0,レノバ,電気・ガス,プライム


In [8]:
### 直近営業日から1日前、1週間前、1か月前の営業日を取得。該当日が休業日の場合は、該当日より前の最新営業日を取得
date_1_matket_day_ago = max([date for date in market_date_last_365days if date <= newest_market_date - timedelta(days=1)])
date_1_matket_week_ago = max([date for date in market_date_last_365days if date <= newest_market_date - timedelta(weeks=1)])
date_1_matket_month_ago = max([date for date in market_date_last_365days if date <= newest_market_date - relativedelta(months=1)])

print(date_1_matket_day_ago, date_1_matket_week_ago, date_1_matket_month_ago)

2026-04-30 2026-04-24 2026-04-01


In [9]:
### 1日前、1週間前、1か月前の営業日の終値を取得
daily_stock_df_1day_ago = (
    daily_stock_df[daily_stock_df["Date"] == date_1_matket_day_ago][["Code", "AdjC"]]
    .rename(columns={"AdjC":"AdjC_1day_ago"})
    .reset_index(drop=True)
)
daily_stock_df_1week_ago = (
    daily_stock_df[daily_stock_df["Date"] == date_1_matket_week_ago][["Code", "AdjC"]]
    .rename(columns={"AdjC":"AdjC_1week_ago"})
    .reset_index(drop=True)
)
daily_stock_df_1month_ago = (
    daily_stock_df[daily_stock_df["Date"] == date_1_matket_month_ago][["Code", "AdjC"]]
    .rename(columns={"AdjC":"AdjC_1month_ago"})
    .reset_index(drop=True)
)

daily_stock_df_1week_ago

,Code,AdjC_1week_ago
0,13010,4640.0
1,13050,3980.0
2,13060,393.7
3,13080,3889.0
4,13090,54500.0
...,...,...
4448,99910,1630.0
4449,99930,1154.0
4450,99940,2237.0
4451,99960,2262.0


In [31]:
### 新高値更新銘柄テーブルに、1日前、1週間前、1か月前の株価終値を追加し、伸び率を計算
new_highs_df = pd.merge(new_highs_df, daily_stock_df_1day_ago, how="left", on="Code")
new_highs_df = pd.merge(new_highs_df, daily_stock_df_1week_ago, how="left", on="Code")
new_highs_df = pd.merge(new_highs_df, daily_stock_df_1month_ago, how="left", on="Code")

new_highs_df["Growth_rate_1day"] = (new_highs_df["AdjC"] / new_highs_df["AdjC_1day_ago"]) * 100 - 100
new_highs_df["Growth_rate_1week"] = (new_highs_df["AdjC"] / new_highs_df["AdjC_1week_ago"]) * 100 - 100
new_highs_df["Growth_rate_1month"] = (new_highs_df["AdjC"] / new_highs_df["AdjC_1month_ago"]) * 100 - 100

# 3市場のみに限定
target_Mkt = ["プライム", "スタンダード", "グロース"]
new_highs_df = new_highs_df[new_highs_df["MktNm"].isin(target_Mkt)]

cat_cols = ["Date", "Code", "CoName", "S17Nm", "MktNm"]
num_cols = ["AdjH", "AdjC", "AdjC_1day_ago", "Growth_rate_1day", "AdjC_1week_ago", "Growth_rate_1week", "AdjC_1month_ago", "Growth_rate_1month"]
new_highs_df = new_highs_df[cat_cols + num_cols].sort_values(by=["Growth_rate_1month"], ascending=[False]).reset_index(drop=True)
new_highs_df

,Date,Code,CoName,S17Nm,MktNm,AdjH,AdjC,AdjC_1day_ago,Growth_rate_1day,AdjC_1week_ago,Growth_rate_1week,AdjC_1month_ago,Growth_rate_1month
0,2026-05-01,278A0,Ｔｅｒｒａ Ｄｒｏｎｅ,電機・精密,グロース,10770.0,10770.0,9270.0,16.181230,7840.0,37.372449,4495.0,139.599555
1,2026-05-01,32940,イーグランド,不動産,スタンダード,4850.0,4850.0,4845.0,0.103199,4840.0,0.206612,2316.0,109.412781
2,2026-05-01,485A0,パワーエックス,電機・精密,グロース,11680.0,10480.0,10690.0,-1.964453,7740.0,35.400517,5330.0,96.622889
3,2026-05-01,34490,テクノフレックス,建設・資材,スタンダード,4925.0,4860.0,4785.0,1.567398,4350.0,11.724138,2551.0,90.513524
4,2026-05-01,72360,ティラド,自動車・輸送機,プライム,15900.0,14600.0,11110.0,31.413141,8150.0,79.141104,8230.0,77.399757
...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,2026-05-01,79760,三菱鉛筆,情報通信・サービスその他,プライム,2580.0,2547.0,2334.0,9.125964,2368.0,7.559122,2422.0,5.161024
63,2026-05-01,90410,近鉄グループホールディングス,運輸・物流,プライム,3593.0,3453.0,3333.0,3.600360,3372.0,2.402135,3302.0,4.572986
64,2026-05-01,73170,松屋アールアンドディ,自動車・輸送機,グロース,1100.0,1099.0,1098.0,0.091075,1097.0,0.182315,1095.0,0.365297
65,2026-05-01,61970,ソラスト,情報通信・サービスその他,プライム,1119.0,1119.0,1119.0,0.000000,1117.0,0.179051,1116.0,0.268817


In [ ]:
### 新高値更新銘柄の52週チャートを出力
def save_price_chart(df, output_path):
    plt.figure(figsize=(8, 4))
    plt.plot(df["Date"], df["AdjC"])
    plt.title("Stock Price Chart")
    plt.xlabel("Date")
    plt.ylabel("Stock Price")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

charts_output_dir = Path("../output/charts")
charts_output_dir.mkdir(parents=True, exist_ok=True)

# 出来高0の日は四本値が欠損しているため、スキップして可視化
daily_stock_df = daily_stock_df.dropna(subset=["AdjC"])

new_highs_list = list(new_highs_df["Code"].unique())
groups = [
    (code, df.sort_values("Date"))
    for code, df in daily_stock_df.groupby("Code")
    if code in new_highs_list
]

Parallel(n_jobs=-1)(
    delayed(save_price_chart)(
        df,
        f"{charts_output_dir}/{code}.png"
    )
    for code, df in groups
)

# for code, df in groups:
#     save_price_chart(df, f"{charts_output_dir}/{code}.png")

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

### レポートを作成

In [35]:
### HTML形式で保存
env = Environment(loader=FileSystemLoader("../input/report_templates"))
template = env.get_template("new_high_report.html")

html = template.render(
    report_date=newest_market_date,
    stocks=new_highs_df.to_dict(orient="records")
)

with open(f"../output/reports/new_high_report_{newest_market_date}.html", "w", encoding="utf-8") as f:
    f.write(html)

In [ ]:
### PDF形式に変換
html_to_pdf(
    f"../output/reports/new_high_report_{newest_market_date}.html",
    f"../output/reports/new_high_report_{newest_market_date}.pdf"
) # 手動で代替中

In [16]:
### PDFをクラウド上にアップロードしてURLを取得
# 手動実行

report_url = "https://drive.google.com/file/d/1BHvfKY5mfzvwMy0U7zYa-jU3iApthJm2/view?usp=sharing"

### LINEに通知

In [19]:
### 加工したデータから送信するテキストを作成

all_len = len(new_highs_df)
prime_len = len(new_highs_df[new_highs_df["MktNm"] == "プライム"])
standard_len = len(new_highs_df[new_highs_df["MktNm"] == "スタンダード"])
growth_len = len(new_highs_df[new_highs_df["MktNm"] == "グロース"])

lines = []

lines.append(f"{newest_market_date}の新高値更新銘柄レポートをお届けします。")
lines.append("")
lines.append(f"【プライム市場】 {prime_len}銘柄")
lines.append(f"【スタンダード市場】 {standard_len}銘柄")
lines.append(f"【グロース市場】 {growth_len}銘柄")
lines.append("")
lines.append(f"【3市場合計】 {all_len}銘柄")

lines.append("")
lines.append("Let's テンバガー！")

lines.append("")
lines.append(f"{report_url}")

# # プライム市場の新高値更新銘柄
# lines.append("【プライム市場】")
# prime_df = new_highs_df[new_highs_df["MktNm"] == "プライム"]
# for row in prime_df.itertuples():
#     lines.append(f"・{row.CoName}_{row.Code}")
#     lines.append(f"({row.Growth_rate_1month:+.2f}/{row.Growth_rate_1week:+.2f}/{row.Growth_rate_1day:+.2f})")

# lines.append("")

# # スタンダード市場の新高値更新銘柄
# lines.append("【スタンダード市場】")
# prime_df = new_highs_df[new_highs_df["MktNm"] == "スタンダード"]
# for row in prime_df.itertuples():
#     lines.append(f"・{row.CoName}_{row.Code}")
#     lines.append(f"({row.Growth_rate_1month:+.2f}/{row.Growth_rate_1week:+.2f}/{row.Growth_rate_1day:+.2f})")

# lines.append("")

# # グロース市場の新高値更新銘柄
# lines.append("【グロース市場】")
# prime_df = new_highs_df[new_highs_df["MktNm"] == "グロース"]
# for row in prime_df.itertuples():
#     lines.append(f"・{row.CoName}_{row.Code}")
#     lines.append(f"({row.Growth_rate_1month:+.2f}/{row.Growth_rate_1week:+.2f}/{row.Growth_rate_1day:+.2f})")
    
# # メッセージリストを文字列に変換
message = "\n".join(lines)

In [36]:
### LINEに通知
headers = {
    "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

payload = {
    "messages": [
        {
            "type": "text",
            "text": message
        }        
    ]
}

res = requests.post(line_base_url + Path_broadcast, headers=headers, json=payload)

print(res.status_code)
print(res.text)

res.raise_for_status()

200
{}
